# Multi-Model Oxidation-kp Regression

Same pipeline as `GBR.ipynb`, but the model is chosen with a single variable
(`MODEL_NAME`) and Optuna automatically uses the matching hyperparameter search
space.

**Features:** raw steel composition (varying element fractions) + test temperature
(`invT`). No external HEA descriptors (they have no formula in this repo).

**Available models** (`MODEL_NAME`):
`gbr`, `histgb`, `randomforest`, `extratrees` (sklearn, ready to use) and
`xgboost`, `lightgbm`, `catboost` (need `pip install xgboost lightgbm catboost`).
Libraries are imported lazily, so an uninstalled one only errors if you select it.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import functions as fs
import optuna
import joblib
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import make_scorer, r2_score, mean_squared_error, mean_absolute_error

plt.close('all')

# ============================ CHOOSE MODEL HERE =============================
MODEL_NAME = 'gbr'   # gbr | histgb | randomforest | extratrees | xgboost | lightgbm | catboost
SEED = 42
N_TRIALS = 100       # number of Optuna trials
# ===========================================================================

file_path = 'docs/Data_base.xlsx'
df = pd.read_excel(file_path, index_col=0)

In [ ]:
# Element columns = everything except the temperature feature (invT) and the target (kp)
element_names = [c for c in df.columns if c not in ('invT', 'kp')]
print('Elemental features (%d):' % len(element_names), element_names)

In [ ]:
# Composition (only elements that vary in the DB) + test temperature (invT).
# Excluded: external HEA descriptors (Smix_HEA, DeltaSize_HEA, Electroneg_HEA,
# Hmix_HEA, Omega_HEA, VEC) and the engineered Al+Cr / Cr/Al.
composition_features = [e for e in element_names if df[e].nunique() > 1] + ['invT']
print('Features used (%d):' % len(composition_features), composition_features)

In [ ]:
trainset, testset = fs.data_split(df, element_names, 0.2)

comp_major_low, comp_major_high, comp_major_inter = -0.1, 100.3, 10
comp_minor_low, comp_minor_high, comp_minor_inter = -0.1, 50.3, 0.1
T_low, T_high, T_inter = 10, 2510, 50
size = 8

# binning args reused by both data_sampling and the leak-free CV (fs.leakfree_cv)
sampling_params = dict(comp_major_low=comp_major_low, comp_major_high=comp_major_high, comp_major_inter=comp_major_inter,
                       comp_minor_low=comp_minor_low, comp_minor_high=comp_minor_high, comp_minor_inter=comp_minor_inter,
                       T_low=T_low, T_high=T_high, T_inter=T_inter, size=size)

Sampled_trainset = fs.data_sampling(trainset, comp_major_low, comp_major_high, comp_major_inter,
                                    comp_minor_low, comp_minor_high, comp_minor_inter,
                                    T_low, T_high, T_inter, size, element_names, random_state=SEED)

In [ ]:
sel_tr = Sampled_trainset.loc[:, Sampled_trainset.columns.intersection(composition_features)]
sel_tr['kp'] = Sampled_trainset['kp']
sel_te = testset.loc[:, testset.columns.intersection(composition_features)]
sel_te['kp'] = testset['kp']

X_train = sel_tr.drop(columns=['kp']); y_train = np.log10(sel_tr['kp'])
X_test  = sel_te.drop(columns=['kp']); y_test  = np.log10(sel_te['kp'])
print('X_train:', X_train.shape, '| X_test:', X_test.shape)

## Model registry

Each entry maps a model name to `(estimator factory, Optuna search-space)`.
To add another model, write an `est_*` factory and a `space_*` function and add a
row to `MODEL_REGISTRY`.

In [ ]:
# ---- sklearn Gradient Boosting -------------------------------------------
def est_gbr(params, seed):
    from sklearn.ensemble import GradientBoostingRegressor
    return GradientBoostingRegressor(**params, random_state=seed)

def space_gbr(trial):
    p = dict(
        n_estimators=trial.suggest_int('n_estimators', 100, 800),
        max_depth=trial.suggest_int('max_depth', 2, 8),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        subsample=trial.suggest_float('subsample', 0.5, 1.0),
        min_samples_split=trial.suggest_int('min_samples_split', 2, 20),
        min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 20),
        max_features=trial.suggest_float('max_features', 0.3, 1.0),
        loss=trial.suggest_categorical('loss', ['squared_error', 'absolute_error', 'huber']),
    )
    if p['loss'] == 'huber':
        p['alpha'] = trial.suggest_float('alpha', 0.75, 0.99)
    return p

# ---- sklearn HistGradientBoosting (fast, native) -------------------------
def est_hgb(params, seed):
    from sklearn.ensemble import HistGradientBoostingRegressor
    return HistGradientBoostingRegressor(**params, random_state=seed)

def space_hgb(trial):
    return dict(
        max_iter=trial.suggest_int('max_iter', 100, 800),
        max_depth=trial.suggest_int('max_depth', 2, 12),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        max_leaf_nodes=trial.suggest_int('max_leaf_nodes', 15, 255),
        l2_regularization=trial.suggest_float('l2_regularization', 1e-8, 10.0, log=True),
        min_samples_leaf=trial.suggest_int('min_samples_leaf', 5, 50),
    )

# ---- sklearn RandomForest / ExtraTrees -----------------------------------
def est_rf(params, seed):
    from sklearn.ensemble import RandomForestRegressor
    return RandomForestRegressor(**params, random_state=seed, n_jobs=-1)

def est_et(params, seed):
    from sklearn.ensemble import ExtraTreesRegressor
    return ExtraTreesRegressor(**params, random_state=seed, n_jobs=-1)

def space_forest(trial):
    return dict(
        n_estimators=trial.suggest_int('n_estimators', 100, 800),
        max_depth=trial.suggest_int('max_depth', 3, 20),
        min_samples_split=trial.suggest_int('min_samples_split', 2, 20),
        min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 20),
        max_features=trial.suggest_float('max_features', 0.3, 1.0),
    )

# ---- XGBoost -------------------------------------------------------------
def est_xgb(params, seed):
    from xgboost import XGBRegressor
    return XGBRegressor(**params, random_state=seed, n_jobs=-1,
                        objective='reg:squarederror', verbosity=0)

def space_xgb(trial):
    return dict(
        n_estimators=trial.suggest_int('n_estimators', 100, 800),
        max_depth=trial.suggest_int('max_depth', 2, 10),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        subsample=trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.3, 1.0),
        min_child_weight=trial.suggest_int('min_child_weight', 1, 20),
        reg_alpha=trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        gamma=trial.suggest_float('gamma', 1e-8, 5.0, log=True),
    )

# ---- LightGBM ------------------------------------------------------------
def est_lgbm(params, seed):
    from lightgbm import LGBMRegressor
    return LGBMRegressor(**params, random_state=seed, n_jobs=-1,
                         subsample_freq=1, verbose=-1)

def space_lgbm(trial):
    return dict(
        n_estimators=trial.suggest_int('n_estimators', 100, 800),
        num_leaves=trial.suggest_int('num_leaves', 15, 255),
        max_depth=trial.suggest_int('max_depth', -1, 12),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        subsample=trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.3, 1.0),
        min_child_samples=trial.suggest_int('min_child_samples', 5, 50),
        reg_alpha=trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    )

# ---- CatBoost ------------------------------------------------------------
def est_cat(params, seed):
    from catboost import CatBoostRegressor
    return CatBoostRegressor(**params, random_state=seed, verbose=0)

def space_cat(trial):
    return dict(
        iterations=trial.suggest_int('iterations', 100, 800),
        depth=trial.suggest_int('depth', 2, 10),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        l2_leaf_reg=trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        subsample=trial.suggest_float('subsample', 0.5, 1.0),
    )

MODEL_REGISTRY = {
    'gbr':          (est_gbr,  space_gbr),
    'histgb':       (est_hgb,  space_hgb),
    'randomforest': (est_rf,   space_forest),
    'extratrees':   (est_et,   space_forest),
    'xgboost':      (est_xgb,  space_xgb),
    'lightgbm':     (est_lgbm, space_lgbm),
    'catboost':     (est_cat,  space_cat),
}

assert MODEL_NAME in MODEL_REGISTRY, \
    'Unknown MODEL_NAME=%r; choose from %s' % (MODEL_NAME, list(MODEL_REGISTRY))
make_estimator, suggest_space = MODEL_REGISTRY[MODEL_NAME]
print('Selected model:', MODEL_NAME)

In [ ]:
# Fail early (before the long Optuna run) if the selected model's library is missing
import importlib.util
_REQUIRED_PKG = {'xgboost': 'xgboost', 'lightgbm': 'lightgbm', 'catboost': 'catboost'}
_pkg = _REQUIRED_PKG.get(MODEL_NAME)
if _pkg and importlib.util.find_spec(_pkg) is None:
    raise ImportError("Model '%s' needs the '%s' package. Install it with:  pip install %s"
                      % (MODEL_NAME, _pkg, _pkg))
print('Library check OK for:', MODEL_NAME)

In [ ]:
# Optuna tuning with leak-free CV (resampling happens INSIDE each fold, on train rows only)
# Objective: MAXIMISE R2. MSE and MAE are kept per-trial (user_attr) for cross-checking.
def objective(trial):
    params = suggest_space(trial)
    cv = fs.leakfree_cv(lambda: make_estimator(params, SEED),
                        trainset, element_names, composition_features,
                        sampling_params, n_splits=5, seed=SEED)
    trial.set_user_attr('MSE', cv['test_MSE'].mean())   # kept for checking
    trial.set_user_attr('MAE', cv['test_MAE'].mean())   # kept for checking
    return cv['test_R2'].mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=N_TRIALS)
print('Best CV R2 :', study.best_value)
print('Best CV MSE:', study.best_trial.user_attrs['MSE'])
print('Best CV MAE:', study.best_trial.user_attrs['MAE'])
print('Best params:', study.best_params)

In [ ]:
best_params = study.best_params
reg = make_estimator(best_params, SEED)

# Leak-free CV report (resampling inside each fold)
cv = fs.leakfree_cv(lambda: make_estimator(best_params, SEED),
                    trainset, element_names, composition_features,
                    sampling_params, n_splits=5, seed=SEED)
print('Train CV MAE:', cv['test_MAE'].mean(), '+/-', cv['test_MAE'].std())
print('Train CV MSE:', cv['test_MSE'].mean(), '+/-', cv['test_MSE'].std())
print('Train CV R2 :', cv['test_R2'].mean(),  '+/-', cv['test_R2'].std())

reg.fit(X_train, y_train)
y_test_pred = reg.predict(X_test)
print('Test MAE:', mean_absolute_error(y_test, y_test_pred))
print('Test MSE:', mean_squared_error(y_test, y_test_pred))
print('Test R2 :', r2_score(y_test, y_test_pred))

In [ ]:
y_train_pred = reg.predict(X_train)
y_test_pred = reg.predict(X_test)

plt.figure(figsize=(7, 7))
sns.set(style='darkgrid')
plt.scatter(y_train, y_train_pred, label='Training Set')
plt.scatter(y_test, y_test_pred, label='Test Set')
plt.plot([-16, -2], [-16, -2], linestyle='--', color='black')
plt.ylim(-16, -2); plt.xlim(-16, -2)
plt.ylabel('Prediction Oxidation Rate Constant log[$k_p$] $(g^2cm^{-4}s^{-1})$')
plt.xlabel('True Oxidation Rate Constant log[$k_p$] $(g^2cm^{-4}s^{-1})$')
plt.title('%s  (Test R2 = %.3f)' % (MODEL_NAME, r2_score(y_test, y_test_pred)))
plt.legend()
plt.show()

In [ ]:
# ---- Feature importance: permutation importance on the held-out test set (model-agnostic) ----
# Works for every model in the registry (tree / boosting / etc.): for each feature it measures
# how much the test R2 drops when that single feature's values are shuffled.
from sklearn.inspection import permutation_importance

perm = permutation_importance(reg, X_test, y_test, n_repeats=20, random_state=SEED, scoring='r2')
importance = pd.DataFrame({'feature': X_test.columns,
                           'importance': perm.importances_mean,
                           'std': perm.importances_std}).sort_values('importance', ascending=False)
print(importance.to_string(index=False))

order = importance.sort_values('importance')  # ascending so the largest ends up on top
plt.figure(figsize=(7, max(4, 0.3 * len(order))))
plt.barh(order['feature'], order['importance'], xerr=order['std'])
plt.xlabel('Permutation importance (mean R2 drop on test set)')
plt.title('Feature importance -- %s' % MODEL_NAME)
plt.tight_layout()
plt.show()

In [ ]:
# Retrain on the entire database and save the model
Sampled_df = fs.data_sampling(df, comp_major_low, comp_major_high, comp_major_inter,
                              comp_minor_low, comp_minor_high, comp_minor_inter,
                              T_low, T_high, T_inter, size, element_names, random_state=SEED)
sel_full = Sampled_df.loc[:, Sampled_df.columns.intersection(composition_features)]
sel_full['kp'] = Sampled_df['kp']
X_full = sel_full.drop(columns=['kp']); y_full = np.log10(sel_full['kp'])

reg.fit(X_full, y_full)
# Save with the algorithm name in the filename so different models don't overwrite each other
out_path = '%s.pkl' % MODEL_NAME
joblib.dump(reg, out_path)
print('Saved %s model to %s' % (MODEL_NAME, out_path))

# Predict a new composition (composition + temperature only):
#   import functions as fs, joblib
#   model = joblib.load(out_path)
#   log_kp, kp = fs.predict_composition({'Ni':50,'Cr':20,'Co':12,'Al':12,'Fe':6},
#                                       model, temperature_C=1150)